# Generate your requirements.yaml file

This notebook exports the current environment's package requirements to a `requirements.yaml` file and validates existing ones. This file is really important as you will need it to upload it together with the notebook into your repository. Please, run the code cell bellow and follow the steps.

> **IMPORANT**: Make sure that you are running this notebook in the same environment you used to run the notebook you are uploading.

In [ ]:
# @markdown Run this cell to choose notebook and extract requirements

# First of all check if ipywidgets is installed, if not through an informative error
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using 'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

# For the requirements extraction we will need nbformat and pyyaml installed, also check for them
try:
    import nbformat
except ImportError:
    raise ImportError("nbformat is not installed. Please install it using 'pip install nbformat' or 'conda install -c conda-forge nbformat'.")

try:
    import yaml
except ImportError:
    raise ImportError("pyyaml is not installed. Please install it using 'pip install pyyaml' or 'conda install -c conda-forge pyyaml'.")

from IPython.display import display
from pathlib import Path
import importlib.metadata
import subprocess
import contextlib
import importlib
import platform
import nbformat
import yaml
import re
import sys
import os
import io

import_regex = r'^[ \t]*import .*'
from_regex = r'^[ \t]*from .* import .*'

def extract_code_import_lines(code):
    list_import_lines = []

    # We are going line by line analyzing them
    lines = code.split('\n')
    for line in lines:
        if re.match(import_regex, line) or re.match(from_regex, line):
            list_import_lines.append(line.strip())
    return list_import_lines

def extract_imported_packages(code):
    list_import_packages = []
    lines = code.split('\n')
    for line in lines:
        line = line.strip()

        # Handle "import X" or "import X, Y" or "import X as Y"
        if re.match(import_regex, line):
            # Extract package name after 'import'
            match = re.search(r'import\s+(.+)', line)
            if match:
                packages = match.group(1).split(',')  # Handle multiple imports
                for pkg in packages:
                    pkg = pkg.split(' as ')[0].strip()  # Remove "as alias"
                    pkg = pkg.split('.')[0]  # Get root package only
                    if pkg:
                        list_import_packages.append(pkg)

        # Handle "from X import Y"
        elif re.match(from_regex, line):
            # Extract module name after 'from'
            match = re.search(r'from\s+([^\s]+)\s+import', line)
            if match:
                module = match.group(1).split('.')[0]  # Get root package
                if module and module != '.':  # Skip relative imports
                    list_import_packages.append(module)

    return list_import_packages

def extract_pip_install(code):
    list_pip_install_lines = []
    lines = code.split('\n')
    
    for line in lines:
        line = line.strip()
        
        # Skip empty lines and comments
        if not line or line.startswith('#'):
            continue
        
        # Check if line starts with ! or %
        if line.startswith('!') or line.startswith('%'):
            # Remove the leading ! or %
            clean_command = line[1:].strip()
            
            # Verify it contains 'install' keyword
            if 'install' in clean_command:
                # Remove trailing comments if present
                if '#' in clean_command:
                    clean_command = clean_command.split('#')[0].strip()
                
                list_pip_install_lines.append(clean_command)
    
    return list_pip_install_lines

def is_builtin_or_stdlib(package_name):
    """Check if a package is a built-in or standard library module."""
    if package_name in sys.builtin_module_names:
        return True

    if hasattr(sys, 'stdlib_module_names') and package_name in sys.stdlib_module_names:
        return True

    # Fallback: common stdlib modules for older Python versions
    stdlib_modules = {
        're', 'sys', 'os', 'platform', 'pathlib', 'json', 'math', 'random',
        'datetime', 'time', 'collections', 'itertools', 'functools', 'operator',
        'pickle', 'csv', 'sqlite3', 'threading', 'subprocess', 'shutil', 'glob',
        'io', 'codecs', 'logging', 'argparse', 'configparser', 'tempfile',
        'urllib', 'http', 'email', 'base64', 'hashlib', 'hmac', 'secrets',
        'struct', 'textwrap', 'string', 'warnings', 'contextlib', 'abc',
        'importlib', 'inspect', 'traceback', 'unittest', 'doctest', 'pdb',
        'profile', 'pstats', 'timeit', 'statistics', 'asyncio', 'concurrent',
        'socket', 'ssl', 'select', 'selectors', 'queue', 'multiprocessing',
        'copy', 'types', 'enum', 'dataclasses', 'typing', 'weakref',
        'array', 'heapq', 'bisect', 'graphlib', 'decimal', 'fractions',
        'numbers', 'cmath', 'zlib', 'gzip', 'bz2', 'lzma', 'tar', 'zipfile',
        'netrc', 'xdrlib', 'plistlib', 'html', 'xml',
        'webbrowser', 'cgi', 'wsgiref', 'uuid', 'socketserver', 'xmlrpc',
    }

    return package_name in stdlib_modules


def is_environment_specific(package_name):
    """Check if a package is environment-specific and shouldn't be in requirements."""
    environment_specific = {
        'google',
        'google.colab',
        'colab',
        'IPython',
        'ipykernel',
        'jupyter',
        'jupyterlab',
    }
    return package_name in environment_specific


def package_exists_on_pypi(package_name):
    """Check if a package exists on PyPI using the PyPI JSON API."""
    try:
        import requests
        response = requests.get(f"https://pypi.org/pypi/{package_name}/json", timeout=2)
        return response.status_code == 200
    except:
        return True


def get_package_version(package_name):
    try:
        module = importlib.import_module(package_name)
        return module.__version__
    except AttributeError:
        try:
            return importlib.metadata.version(package_name)
        except:
            return ""
    except:
        return ""


def get_package_name_for_pip(import_name):
    """
    Convert import name to pip package name.
    E.g., 'docx' -> 'python-docx', 'cv2' -> 'opencv-python'
    """
    known_mappings = {
        'docx': 'python-docx',
        'cv2': 'opencv-python',
        'skimage': 'scikit-image',
        'PIL': 'Pillow',
        'yaml': 'pyyaml',
        'sklearn': 'scikit-learn',
        'bs4': 'beautifulsoup4',
        'dotenv': 'python-dotenv',
        'fpdf': 'fpdf2',
    }

    if import_name in known_mappings:
        return known_mappings[import_name]

    # Try to find via installed distributions
    try:
        for dist in importlib.metadata.distributions():
            try:
                top_level = dist.read_text('top_level.txt')
                if top_level and import_name in top_level.split('\n'):
                    return dist.name
            except:
                pass
    except:
        pass

    # Fallback: assume import name is correct
    return import_name


def load_pip_freeze_requirements():
    mapping = {}
    try:
        result = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
    except Exception:
        return mapping

    for raw_line in result.splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue

        requirement = line
        key = None

        if line.startswith('-e '):
            entry = line[3:].strip()
            if '#egg=' in entry:
                key = entry.split('#egg=', 1)[1]
            elif ' @ ' in entry:
                key = entry.split(' @ ', 1)[0]
            requirement = entry

        elif ' @ ' in line:
            key = line.split(' @ ', 1)[0]

        elif '==' in line:
            key = line.split('==', 1)[0]

        elif '#egg=' in line:
            key = line.split('#egg=', 1)[1]

        else:
            key = line

        if key:
            mapping[key.strip().lower()] = requirement

    return mapping


def requirement_from_pip_freeze(import_name, pip_freeze_requirements):
    lower_name = import_name.lower()
    if lower_name in pip_freeze_requirements:
        return pip_freeze_requirements[lower_name]

    pip_name = get_package_name_for_pip(import_name)
    if pip_name and pip_name.lower() in pip_freeze_requirements:
        return pip_freeze_requirements[pip_name.lower()]
 
    return None

def sanitize_path_input(raw):
    """
    Turn a user-pasted path into a safe Path without using `os`.
    - Strips ASCII + smart quotes and surrounding punctuation
    - Expands ~ to the user home directory
    - Replaces common Windows placeholders (%USERPROFILE%, %HOMEPATH%) with the home dir
    - Normalises mixed slashes
    """
    s = str(raw or "").strip()
    if not s:
        return Path("")

    # Remove all kinds of quotes people paste (ASCII and curly/smart quotes)
    s = re.sub(r'[\"\'\u201c\u201d\u2018\u2019]', "", s)

    # Trim accidental leading/trailing punctuation or commas
    s = s.strip(" \t\n\r,;<>")

    # Expand ~ at the start to the user's home dir
    home = Path.home().as_posix()
    if s == "~":
        s = home
    elif s.startswith("~/") or s.startswith("~\\"):
        s = home + s[1:]

    # Common Windows env placeholders -> expand to home (no os.environ access here)
    # This covers the most common pasted forms like %USERPROFILE%\Documents
    s = s.replace("%USERPROFILE%", home)
    s = s.replace("%HOMEPATH%", home)

    # Replace repeated backslash-quote artefacts that sometimes appear from copy/paste
    s = s.replace('\\"', '').replace("\\'", "")

    # Normalise forward/back slashes to the OS-agnostic form; Path() will handle exact OS semantics
    s = s.replace("/", str(Path.home()).startswith("\\") and "\\" or "/") if False else s  # noop kept explicit: Path handles slashes

    # Final strip (in case replacements introduced new edge whitespace/punctuation)
    s = s.strip(" \t\n\r,;<>")

    return Path(s)

def is_colab():
    return "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ


def pick_notebook_path_ui_then_run(colab_flag):
    """
    In Colab, ask the user for a notebook path using widgets (recommended: Drive path).
    When a valid path is provided and confirmed, run the extraction.
    """
    mount_title = widgets.HTML(
        "<h2>Step 1: Click the button to mount your Google Drive</h2>"
    )
    mount_btn = widgets.Button(
        description="Mount Google Drive",
        button_style="info",
        layout=widgets.Layout(width="100%")
    )
    mount_status = widgets.HTML("")

    colab_find_notebook = widgets.HTML(
        "<h2>Step 2: Find your notebook</h2>"
        "<ol>"
        "<li>On the left sidebar, click on the folder icon to open the file explorer.</li>"
        "<li>You should see a `drive` folder, otherwise go back to Step 1.</li>"
        "<li>Go to <b>drive</b> > <b>MyDrive</b>. There you will find your Google Drive files.</li>"
        "<li>Locate the notebook you want to extract requirements from. Notebook are stored by default in <b>Colab Notebooks</b> folder.</li>"
        "</ol>"

        "<h4>What if I cannot find my notebook?</h4>"
        "It might happen that your notebook is not stored in Google Drive. <b>Do you have it stored locally on your computer?</b>"
        "<ul>"
        "<li>If <b>yes</b>, drag and drop it in Colab's file explorer for a single use or upload it to Google Drive for persistent storage.</li>"
        "<li>If <b>not</b>, you can download it from wherever you have it stored (e.g., GitHub, email, etc.) and then upload it to Google Drive or directly to Colab.</li>"
        "</ul>"
        "In case the notebook is in another Colab session you can always download it following these steps:"
        "<ol>"
        "<li>Open the Colab session where your notebook is.</li>"
        "<li>On top, go to <b>File</b> > <b>Download</b> > <b>Download .ipynb</b>.</li>"
        "</ol>"
    )
    local_find_notebook = widgets.HTML(
        "<h2>Step 1: Locate your local notebook</h2>"
        "Please make sure you know the full path to your local notebook file (with .ipynb extension)."
    )

    colab_path_title = widgets.HTML(
        "<h2>Step 3: Provide a notebook path</h2>"
        "Right-click on the notebook on Colab's file explorer and select 'Copy path'.<br>"
        "Then paste the full path below and click 'Use this notebook'."
    )
    local_path_title = widgets.HTML(
        "<h2>Step 2: Provide a notebook path</h2>"
        "Please paste the full path to your local notebook below and click 'Use this notebook'."
    )
    path_widget = widgets.Text(
        value="/content/drive/MyDrive/your_notebook.ipynb" if colab_flag else "path/to/your_notebook.ipynb",
        description="Notebook path:",
        layout=widgets.Layout(width="99%"),
        style={'description_width': 'initial'}
    )
    path_status = widgets.HTML("")
    results_output = widgets.Output()

    use_btn = widgets.Button(
        description="Use this notebook",
        button_style="success",
        layout=widgets.Layout(width="99%")
    )

    def on_mount(_):
        try:
            from google.colab import drive
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                drive.mount("/content/drive", force_remount=True)
            mount_status.value = "<b style='color:green;'>Drive mounted.</b> Update the path if needed, then click 'Use this notebook'."
        except Exception as e:
            mount_status.value = f"<b style='color:red;'>Drive mount failed:</b> {str(e)}"

    def on_use(_):
        nb_path = sanitize_path_input(path_widget.value).expanduser()

        # Allow folder input: if it's a directory, try to find .ipynb inside it
        if nb_path.exists() and nb_path.is_dir():
            nbs = sorted(nb_path.glob("*.ipynb"))
            if len(nbs) == 1:
                nb_path = nbs[0]
            elif len(nbs) > 1:
                path_status.value = "<b style='color:red;'>Multiple notebooks found in that folder. Please paste the full .ipynb path.</b>"
                return
            else:
                path_status.value = "<b style='color:red;'>No .ipynb files found in that folder. Please paste a full .ipynb path.</b>"
                return

        if not nb_path.exists() or nb_path.suffix != ".ipynb":
            if colab_flag:
                path_status.value = (
                    "<b style='color:red;'>Invalid path.</b> Please paste a valid .ipynb path and try again, e.g., "
                    "<code>/content/drive/MyDrive/Colab Notebooks/your_notebook.ipynb</code>"
                )
                return
            else:
                path_status.value = (
                    "<b style='color:red;'>Invalid path.</b> Please paste an absolute path to a valid local .ipynb path and try again. For example: "
                    "<ul>"
                    r"<li><code>C:\Users\YourName\Notebooks\your_notebook.ipynb</code> (Windows)</li>"
                    "<li><code>/home/yourname/notebooks/your_notebook.ipynb</code> (Linux/Mac)</li>"
                    "</ul>"
                )
                return

        path_status.value = f"<b style='color:green;'>Using:</b> <code>{nb_path}</code>"
        results_output.clear_output()
        with results_output:
            run_requirements_extraction(nb_path)

    mount_btn.on_click(on_mount)
    use_btn.on_click(on_use)

    if colab_flag:
        display(mount_title, mount_btn, mount_status, colab_find_notebook,
                colab_path_title, path_widget, use_btn, path_status, results_output)
    else:
        display(local_find_notebook, local_path_title, path_widget, use_btn, path_status, results_output)

def run_requirements_extraction(notebook_path: Path):
    # Load pip-freeze mapping (do it here so it reflects the runtime env at click-time)
    pip_freeze_requirements = load_pip_freeze_requirements()

    notebook_name = notebook_path.name

    # Read the original notebook
    colab_nb = nbformat.read(notebook_path, as_version=4)

    # Gather imports
    all_imports = []
    pip_install_lines = []
    for cell in colab_nb.cells:
        if cell.cell_type == "code":
            all_imports.extend(extract_imported_packages(cell.source))
            pip_install_lines.extend(extract_pip_install(cell.source))

    # Remove duplicates while preserving order
    all_imports = list(dict.fromkeys(all_imports))

    # Process packages to get versions
    versioned_packages = []
    not_found_packages = []
    filtered_packages = []

    for pkg in all_imports:
        normalized_pkg = pkg.strip()

        if is_builtin_or_stdlib(normalized_pkg):
            continue

        if is_environment_specific(normalized_pkg):
            filtered_packages.append(f"{normalized_pkg} (environment-specific)")
            continue

        freeze_requirement = requirement_from_pip_freeze(normalized_pkg, pip_freeze_requirements)
        pip_name = get_package_name_for_pip(normalized_pkg)

        if freeze_requirement:
            versioned_packages.append(freeze_requirement)
            continue

        try:
            version = get_package_version(normalized_pkg)
        except Exception:
            version = ""

        if not version and pip_name and pip_name != normalized_pkg:
            version = get_package_version(pip_name)

        if pip_name and not package_exists_on_pypi(pip_name):
            filtered_packages.append(f"{pip_name} (not on PyPI)")
            continue

        resolved_name = pip_name or normalized_pkg

        if version:
            versioned_packages.append(f"{resolved_name}=={version}")
        else:
            not_found_packages.append(resolved_name)

    # Get the current Python version
    python_version = platform.python_version()

    def show_requirements_dialog():
        """Display the version info and generate requirements dialog."""
        version_info = widgets.HTML(
            "<h2>Extracted Requirements</h2>"
            "<p>The following packages were detected from the notebook imports.</p>"
        )
        # Print found packages and not found packages
        version_info.value += "<b>Found packages with versions:</b><br>"
        version_info.value += "<ul>"
        for vp in versioned_packages:
            version_info.value += f"<li>{vp}</li>"
        version_info.value += "</ul>"

        if filtered_packages:
            version_info.value += "<b>Filtered packages (excluded from requirements):</b><br>"
            version_info.value += "<ul>"
            for fp in filtered_packages:
                version_info.value += f"<li>{fp}</li>"
            version_info.value += "</ul>"

        if not_found_packages:
            version_info.value += "<b>Packages not found on your environment or without version info:</b><br>"
            version_info.value += "<ul>"
            for np in not_found_packages:
                version_info.value += f"<li>{np}</li>"
            version_info.value += "</ul>"

        version_info.value += f"<b>Current Python version:</b> {python_version}<br>"

        # Ask for a notebook description + generate requirements.yaml
        default_notebook_description = f"This notebook '{notebook_name}' is for ..."

        description = widgets.Textarea(
            value=default_notebook_description,
            placeholder='Type something',
            description='Description:',
            disabled=False,
            layout=widgets.Layout(width='100%', height='80px')
        )

        button = widgets.Button(
            description='Generate requirements.yaml',
            disabled=False,
            button_style='success',
            tooltip='Click to generate requirements.yaml file',
            icon='check',
            layout=widgets.Layout(width='100%')
        )
        output = widgets.HTML()

        def on_button_clicked(_):
            if not description.value.strip():
                output.value = "<b style='color:red;'>Please provide a valid description.</b>"
                return

            requirements = {
                'description': description.value,
                'python_version': python_version,
                'dependencies': versioned_packages,
            }

            with open('requirements.yaml', 'w') as file:
                yaml.dump(requirements, file, default_flow_style=False)

            output.value = "<b>requirements.yaml file has been generated successfully!</b>"
            output.value += "Please validate its content, edit if necessary (missed packages or edge cases) and in case you edited validate it on the following code cell."


        button.on_click(on_button_clicked)

        display(version_info, description, button, output)

    # Check if there are pip install lines
    if pip_install_lines:
        explanation_html = widgets.HTML(
            "<b>Note:</b> The following pip install commands were detected in the notebook.<br>"
            "Please ensure these packages are installed in order to be detected and included in the generated requirements.yaml file.<br>"
            "If you want to install them now, please:"
            "<ol>" 
            "<li>Copy and paste the code lines bellow.</li>"
            "<li>Create a new code cell and paste the code lines there.</li>"
            "<li>Run the cell to install the packages.</li>"
            "<li>Re-run this requirements extraction cell after installation.</li>"
            "</ol>"
            "If you have already installed them, you can ignore this message and click on the 'Continue to generate requirements.yaml' button below."
        )
        pip_installed_textarea = widgets.Textarea(
            value="\n".join([f"!{line}" for line in pip_install_lines]),
            description='Detected pip installs:',
            layout=widgets.Layout(width='100%', height='100px'),
            style={'description_width': 'initial'}
        )
        continue_button = widgets.Button(
            description='Continue to generate requirements.yaml',
            disabled=False,
            button_style='success',
            tooltip='Click to continue',
            icon='check',
            layout=widgets.Layout(width='100%')
        )
        
        def on_continue_clicked(_):
            # Don't close widgets, just hide the pip install section and show requirements dialog
            explanation_html.layout = widgets.Layout(display='none')
            pip_installed_textarea.layout = widgets.Layout(display='none')
            continue_button.layout = widgets.Layout(display='none')
            show_requirements_dialog()
        
        continue_button.on_click(on_continue_clicked)
        display(explanation_html, pip_installed_textarea, continue_button)
    else:
        # No pip installs found, show requirements dialog directly
        show_requirements_dialog()

# ------------------ ENTRYPOINT ------------------
pick_notebook_path_ui_then_run(is_colab())

In [ ]:
# @markdown Validate your requirements.yaml file

import yaml
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

required_keys = {"dependencies", "python_version", "description"}
alt_python_key = "python"

def validate_requirements(path: Path):
    if not path.exists():
        return False, [f"File not found: {path}"]
    try:
        with path.open("r", encoding="utf-8") as f:
            data = yaml.safe_load(f)
    except Exception as exc:
        return False, [f"YAML read error: {exc}"]
    if not isinstance(data, dict):
        return False, ["YAML root must be a mapping (key/value pairs)."]

    keys = set(data.keys())
    missing = required_keys - keys
    messages = []
    python_key = "python_version"
    if "python_version" not in data and alt_python_key in data:
        python_key = alt_python_key
        missing -= {"python_version"}
    if missing:
        messages.append("Missing required keys: " + ", ".join(sorted(missing)))

    deps = data.get("dependencies")
    if deps is None:
        messages.append("Key 'dependencies' is required.")
    elif not isinstance(deps, list):
        messages.append("'dependencies' must be a YAML list (lines prefixed with '- ').")
    else:
        for d in deps:
            if not isinstance(d, str):
                messages.append(f"{d} (is not a string)")
            elif not d.strip():
                messages.append(f"{d} (is an empty string)")
            elif ' - ' in d:
                messages.append(f"{d} (contains ' - ': possible indentation error in YAML)")

    if python_key not in data:
        messages.append("'python_version' is required.")
    elif not isinstance(data.get(python_key), str) or not data.get(python_key).strip():
        messages.append(f"'{python_key}' must be a non-empty string (for example: 3.10.13).")

    if "description" not in data or not isinstance(data.get("description"), str) or not data.get("description").strip():
        messages.append("'description' must be a non-empty string.")

    return len(messages) == 0, messages

path_input = widgets.Text(
    value="requirements.yaml",
    description="Path:",
    layout=widgets.Layout(width="99%"),
    style={"description_width": "initial"},
)
validate_btn = widgets.Button(
    description="Validate requirements.yaml",
    button_style="success",
    layout=widgets.Layout(width="99%"),
)
output_html = widgets.HTML()

def on_validate(_):
    path = Path(path_input.value).expanduser().resolve()
    ok, messages = validate_requirements(path)
    if ok:
        output_html.value = f"<b>✅ Validation passed for {path.name}.</b>"
    else:
        output_html.value = f"<b>⚠️ Issues found in the given {path.name}:</b><ul>"
        for msg in messages:
            output_html.value += f"<li>❌ {msg}</li>"
        output_html.value += "</ul>"

validate_btn.on_click(on_validate)
display(path_input, validate_btn, output_html)
